[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CkCreative/transformers-and-math-notebooks/blob/main/27-recognition-with-ctc.ipynb)

# Module 27: Recognition: OCR and Speech With CTC

Companion notebook to [Module 27](https://transformers-and-math.drmhse.com/modules/27-recognition-with-ctc) of *Transformers & Math*. Run the cells in order; each part names the section of the module it goes with. Everything runs on a free Colab CPU in under a minute or two.

In [ ]:
import itertools
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)

## Part 1 — The collapsing rule (section 27.2)

In [ ]:
BLANK = '_'
def collapse(path):
    merged = [c for i, c in enumerate(path) if i == 0 or c != path[i - 1]]
    return ''.join(c for c in merged if c != BLANK)

for p in ['cc_at', '_c_aat', 'c_cat', 'bb_oo_ok']:
    print(f'{p:10} → {collapse(p)!r}')

## Part 2 — The CTC loss: brute force against the recursion (section 27.3)

In [ ]:
symbols = [BLANK, 'a', 'c', 't']
T = 6
log_probs = torch.randn(T, len(symbols)).log_softmax(-1)

brute = torch.logsumexp(torch.stack([
    sum(log_probs[t, symbols.index(c)] for t, c in enumerate(path))
    for path in itertools.product(symbols, repeat=T) if collapse(path) == 'cat'
]), 0)
target = torch.tensor([[symbols.index(c) for c in 'cat']])
ctc = F.ctc_loss(log_probs[:, None], target, torch.tensor([T]), torch.tensor([3]), blank=0, reduction='sum')
print(f'brute force over {len(symbols) ** T:,} paths: −log P = {-brute.item():.6f}')
print(f'PyTorch ctc_loss (the recursion):        {ctc.item():.6f}')

## Part 3 — Train a tiny recognizer with no alignments (section 27.3)
Inputs are noisy one-hot "frames" in which each character lasts a random number of frames, with a short silence after it. The model is told only the transcript, never which frames belong to which letter.

In [ ]:
alphabet = 'abcde'
def make(n=64):
    xs, ys, lens = [], [], []
    for _ in range(n):
        word = ''.join(alphabet[i] for i in torch.randint(0, 5, (4,)))
        frames = []
        for c in word:
            frames += [alphabet.index(c) + 1] * int(torch.randint(2, 5, (1,)))
            frames += [0] * int(torch.randint(1, 3, (1,)))                  # silence between letters, so repeats stay separable
        x = F.one_hot(torch.tensor(frames), 6).float() + 0.3 * torch.randn(len(frames), 6)
        xs.append(x); ys.append(torch.tensor([alphabet.index(c) + 1 for c in word])); lens.append(len(frames))
    X = nn.utils.rnn.pad_sequence(xs)                                     # (T, B, 6)
    return X, torch.cat(ys), torch.tensor(lens), torch.full((n,), 4)

class Recognizer(nn.Module):
    """A convolution over neighbouring frames, so each frame's prediction sees a little context."""
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv1d(6, 64, kernel_size=5, padding=2)
        self.out = nn.Linear(64, 6)
    def forward(self, x):                      # x: (T, B, 6)
        h = F.relu(self.conv(x.permute(1, 2, 0))).permute(2, 0, 1)
        return self.out(h)

model = Recognizer()
opt = torch.optim.Adam(model.parameters(), lr=3e-3)
for step in range(600):
    X, Y, xl, yl = make()
    loss = F.ctc_loss(model(X).log_softmax(-1), Y, xl, yl, blank=0)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 150 == 0:
        print(f'step {step:3}  CTC loss {loss.item():.3f}')

X, Y, xl, _ = make(200)
pred = model(X).argmax(-1).T                                              # greedy decode per frame
words = [collapse(''.join(BLANK if i == 0 else alphabet[i - 1] for i in p[:l])) for p, l in zip(pred.tolist(), xl.tolist())]
truth = [''.join(alphabet[i - 1] for i in Y[4 * k:4 * k + 4].tolist()) for k in range(200)]
print(f'exact transcripts: {sum(w == t for w, t in zip(words, truth)) / 200:.0%}   e.g. {words[:5]} vs {truth[:5]}')

## Part 4 — Error rates (section 27.5)

In [ ]:
def edit_distance(a, b):
    d = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        prev, d[0] = d[0], i
        for j, y in enumerate(b, 1):
            prev, d[j] = d[j], min(d[j] + 1, d[j - 1] + 1, prev + (x != y))
    return d[-1]

ref, hyp = 'the quick brown fox jumps over the lazy dog', 'the quick brown fax jumped over a lazy dog'
print(f'CER {edit_distance(ref, hyp) / len(ref):.1%}   WER {edit_distance(ref.split(), hyp.split()) / len(ref.split()):.1%}')

## Your turn
1. Use words that contain double letters (for example `aab`) in Part 3. Does the model learn to output the blank between them?
2. What is the minimum number of frames for "letter"? Check by trying `F.ctc_loss` with shorter inputs (it returns infinity when impossible).